# Explicit FMU port mapping and solver settings

Use this notebook when an FMU has nonstandard electrical variable names, current polarity, or needs importer settings. It runs the same retained inverter FMU used by the quickstart, but spells out the abc mapping and records multirate diagnostics.

In [ ]:
from pathlib import Path
import sys
import numpy as np

ROOT = next(
    path for path in (Path.cwd(), *Path.cwd().parents) if (path / "dpsim-fmi").is_dir()
)
sys.path.insert(0, str(ROOT / "build-fmi"))
import dpsimpy

fmu_path = (
    ROOT
    / "dpsim-fmi/examples/inverter/sources/GridTiedInverterPortedV3.x86_64-linux.fmu"
)
options = dpsimpy.fmi.ModelExchangeOptions()
options.endpoint_state_events = True
settings = dpsimpy.fmi.NewtonSettings()
settings.multirate_mode = dpsimpy.fmi.MultirateMode.SinglePassExplicit

fmu = dpsimpy.fmi.import_fmu_with_mapping(
    str(fmu_path),
    voltage_inputs=["v_a", "v_b", "v_c"],
    current_outputs=["i_a", "i_b", "i_c"],
    current_sign_to_passive=1.0,
    name="mapped_inverter",
    model_options=options,
    settings=settings,
)
print(fmu.description.model_name, fmu.description.can_get_and_set_fmu_state)

The mapping sets the order and current convention explicitly. `SinglePassExplicit` advances an FMU once per macro step and logs the current coupling discrepancy; it does not claim Newton convergence across a macro interval.

In [ ]:
frequency = 50.0
phase_peak = np.sqrt(2.0) * 230.0
angles = np.array([0.0, -2 * np.pi / 3, 2 * np.pi / 3])
source_phasors = np.asarray(
    [phase_peak * np.exp(1j * a) / dpsimpy.RMS3PH_TO_PEAK1PH for a in angles]
).reshape(3, 1)
initial_voltage = [complex(value.real, 0.0) for value in source_phasors[:, 0]]
pcc = dpsimpy.emt.SimNode("pcc", dpsimpy.PhaseType.ABC, initial_voltage)
grid = dpsimpy.emt.ph3.VoltageSource("grid")
grid.set_parameters(source_phasors, frequency)
grid.connect([dpsimpy.emt.SimNode.gnd, pcc])
fmu.connect(pcc)
system = dpsimpy.SystemTopology(frequency, [pcc], [grid, fmu])

results_dir = ROOT / "build-fmi/fmi-python-mapped"
results_dir.mkdir(parents=True, exist_ok=True)
dpsimpy.Logger.set_log_dir(str(results_dir))
logger = dpsimpy.Logger("fmi_python_mapped")
logger.log_attribute("pcc_voltage", pcc.attr("v"))
logger.log_attribute("terminal_current", fmu.attr("i_intf"))
logger.log_attribute("newton_iterations", fmu.attr("newton_iterations"))
logger.log_attribute("internal_steps", fmu.attr("fmi_internal_steps"))
logger.log_attribute("coupling_error_A", fmu.attr("fmi_coupling_error_A"))
simulation = dpsimpy.Simulation("fmi_python_mapped")
simulation.set_domain(dpsimpy.Domain.EMT)
simulation.set_system(system)
simulation.set_time_step(50e-6)
simulation.set_final_time(1e-3)
simulation.do_steady_state_init(False)
simulation.add_logger(logger)
simulation.run()

In [ ]:
import csv

csv_path = results_dir / "fmi_python_mapped.csv"
with csv_path.open(newline="") as result_file:
    reader = csv.DictReader(result_file)
    print("logged columns:", reader.fieldnames)
    for row in [next(reader) for _ in range(3)]:
        print(row)

## Plot terminal magnitudes and comparison errors

The plots show all three phases, vector magnitudes, and importer diagnostics. The quickstart also compares PCC voltage with its ideal source waveform. To compare against another simulation, set a same-case reference CSV and the corresponding column map in the final cell.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Read the simulator log; the short demo is intentionally kept lightweight.
data = pd.read_csv(csv_path, skipinitialspace=True)
data.columns = data.columns.str.strip()
time_ms = data["time"] * 1e3
phase_colors = ["tab:blue", "tab:orange", "tab:green"]


def plot_phases(ax, prefix, ylabel, labels=("a", "b", "c")):
    columns = [f"{prefix}_{k}" for k in range(3)]
    if not all(column in data for column in columns):
        return False
    values = data[columns].to_numpy(dtype=float)
    for k, (column, label) in enumerate(zip(columns, labels)):
        ax.plot(
            time_ms, data[column], color=phase_colors[k], lw=1.1, label=f"phase {label}"
        )
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.25)
    ax.legend(ncol=3, fontsize=8)
    return values


fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
voltage = plot_phases(axes[0], "pcc_voltage", "PCC voltage [V]")
current_name = (
    "converter_current" if "converter_current_0" in data else "terminal_current"
)
current = plot_phases(axes[1], current_name, "Terminal current [A]")
axes[1].set_xlabel("Time [ms]")
fig.suptitle("Three-phase terminal quantities")
fig.tight_layout()
plt.show()

# Vector magnitudes summarize the three logged phase values at each sample.
fig, ax = plt.subplots(figsize=(10, 3.5))
if voltage is not False:
    ax.plot(
        time_ms,
        np.linalg.norm(voltage, axis=1),
        label="PCC voltage vector magnitude [V]",
    )
if current is not False:
    ax.plot(
        time_ms,
        np.linalg.norm(current, axis=1),
        label="terminal current vector magnitude [A]",
    )
ax.set_xlabel("Time [ms]")
ax.set_ylabel("Vector magnitude")
ax.grid(alpha=0.25)
ax.legend()
fig.tight_layout()
plt.show()

# Solver and multirate fields are shown only when the selected importer logs them.
diagnostic_columns = [
    name
    for name in ("newton_iterations", "internal_steps", "coupling_error_A")
    if name in data
]
if diagnostic_columns:
    fig, axes = plt.subplots(
        len(diagnostic_columns),
        1,
        figsize=(10, 2.6 * len(diagnostic_columns)),
        sharex=True,
    )
    axes = np.atleast_1d(axes)
    for ax, name in zip(axes, diagnostic_columns):
        ax.step(time_ms, data[name], where="post", lw=1.1)
        ax.set_ylabel(
            {
                "newton_iterations": "Newton iterations",
                "internal_steps": "FMI steps / macro",
                "coupling_error_A": "Coupling error [A]",
            }[name]
        )
        ax.grid(alpha=0.25)
    axes[-1].set_xlabel("Time [ms]")
    fig.suptitle("Importer diagnostics")
    fig.tight_layout()
    plt.show()

# For a voltage comparison in this quickstart, compare with the ideal source waveform.
# This is an analytic network reference; it is not an independent converter simulation.
if voltage is not False:
    t = data["time"].to_numpy()
    expected_voltage = np.column_stack(
        [phase_peak * np.cos(2 * np.pi * frequency * t + angle) for angle in angles]
    )
    voltage_error = voltage - expected_voltage
    fig, ax = plt.subplots(figsize=(10, 3.5))
    for k, label in enumerate(("a", "b", "c")):
        ax.plot(
            time_ms,
            voltage_error[:, k],
            color=phase_colors[k],
            lw=1.0,
            label=f"phase {label}",
        )
    ax.set_title("PCC voltage difference from the ideal grid source")
    ax.set_xlabel("Time [ms]")
    ax.set_ylabel("Voltage error [V]")
    ax.grid(alpha=0.25)
    ax.legend(ncol=3)
    fig.tight_layout()
    plt.show()
    print(
        f"Ideal-source voltage error: max={np.max(np.abs(voltage_error)):.3g} V, "
        f"RMS={np.sqrt(np.mean(voltage_error**2)):.3g} V"
    )

# Optional independent-simulation comparison. Set a same-case CSV path and map its
# columns to the DPsim column names above; signals are linearly interpolated in time.
# Example: REFERENCE_COLUMNS = {"pcc_voltage_0": "v_pcc_a", "converter_current_0": "i_grid_a"}
REFERENCE_CSV = None
REFERENCE_COLUMNS = {}
if REFERENCE_CSV is not None:
    reference = pd.read_csv(REFERENCE_CSV, skipinitialspace=True)
    reference.columns = reference.columns.str.strip()
    fig, axes = plt.subplots(
        len(REFERENCE_COLUMNS),
        1,
        figsize=(10, 2.8 * len(REFERENCE_COLUMNS)),
        sharex=True,
    )
    axes = np.atleast_1d(axes)
    for ax, (sim_column, ref_column) in zip(axes, REFERENCE_COLUMNS.items()):
        if sim_column not in data or ref_column not in reference:
            raise KeyError(
                f"Missing comparison columns: DPsim={sim_column!r}, reference={ref_column!r}"
            )
        t_ref = reference["time"].to_numpy(dtype=float)
        y_ref = reference[ref_column].to_numpy(dtype=float)
        valid = (data["time"].to_numpy() >= t_ref.min()) & (
            data["time"].to_numpy() <= t_ref.max()
        )
        t_cmp = data.loc[valid, "time"].to_numpy()
        error = data.loc[valid, sim_column].to_numpy() - np.interp(t_cmp, t_ref, y_ref)
        ax.plot(t_cmp * 1e3, error, lw=1.0)
        ax.set_ylabel(f"{sim_column} error")
        ax.grid(alpha=0.25)
        print(
            f"{sim_column}: max error={np.max(np.abs(error)):.4g}, "
            f"RMS error={np.sqrt(np.mean(error**2)):.4g}"
        )
    axes[-1].set_xlabel("Time [ms]")
    fig.suptitle("Difference from independent comparison simulation")
    fig.tight_layout()
    plt.show()
else:
    print(
        "Independent-simulation error plots are optional: set REFERENCE_CSV and REFERENCE_COLUMNS "
        "to a time-aligned, same-case reference dataset."
    )